# Retrieval Latency Benchmark

Runs **only** the retrieval phase (no LLM generation, no answer evaluation) for each
ablation config and records per-question `retrieval_s`.

* Query-entity extraction is **skipped** for questions already in the cache — pure Neo4j latency.
* Results are patched directly into the existing `ablation-results-*/` CSVs so `summary()` picks them up automatically.
* Use `max_workers=1` (default) for accurate serial latency numbers.

In [ ]:
import os
import sys
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

In [ ]:
from qasa_rag.embedder import Embedder
from qasa_rag.retrieval import QASARetriever, AblationConfig, AblationEvaluator

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "2wiki"  # "musique" or "2wiki"
N_EVAL = 500            # number of questions to benchmark
MAX_WORKERS = 1         # 1 = serial (accurate latency); >1 = parallel (throughput)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

In [ ]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

print(f"Loaded {len(ground_truth)} questions for {DATASET_NAME}")

In [ ]:
evaluator = AblationEvaluator(
    driver=driver,
    embedder=embedder,
    ground_truth=ground_truth,
    output_dir=f"ablation-results-{DATASET_NAME}",
    retriever_cls=QASARetriever,
    query_entity_cache_path=Path(f"cache/query_entities-{DATASET_NAME}.pkl"),
)

In [ ]:
configs = {
    "naive_vector": AblationConfig(name="naive-vector", naive_vector=True, top_k_entities=30),
    "no_qa":        AblationConfig(name="no-query-aware", query_aware=False, max_steps=3, decay=0.7),
    "steps_1":      AblationConfig(name="steps-1",  max_steps=1, decay=0.7),
    "steps_2":      AblationConfig(name="steps-2",  max_steps=2, decay=0.7),
    "steps_3":      AblationConfig(name="steps-3",  max_steps=3, decay=0.7),
    "steps_4":      AblationConfig(name="steps-4",  max_steps=4, decay=0.7),
    "decay_03":     AblationConfig(name="decay-0.3", max_steps=3, decay=0.3),
    "decay_05":     AblationConfig(name="decay-0.5", max_steps=3, decay=0.5),
    "decay_07":     AblationConfig(name="decay-0.7", max_steps=3, decay=0.7),
    "decay_09":     AblationConfig(name="decay-0.9", max_steps=3, decay=0.9),
    "decay_10":     AblationConfig(name="decay-1.0", max_steps=3, decay=1.0),
}

## Run benchmark

Run all configs, or just a subset — comment out the ones you don't need.

Each call patches `retrieval_s` in the existing CSV for that config.

In [ ]:
CONFIGS_TO_RUN = list(configs.keys())

timing_results = {}
for key in CONFIGS_TO_RUN:
    timing_results[key] = evaluator.benchmark_retrieval(
        configs[key],
        n_eval=N_EVAL,
        max_workers=MAX_WORKERS,
        patch_csv=True,
    )

## Summary table

In [ ]:
import pandas as pd

rows = []
for key, df in timing_results.items():
    rows.append({
        "config": configs[key].name,
        "n": len(df),
        "avg_retrieval_s": df["retrieval_s"].mean(),
        "p50_retrieval_s": df["retrieval_s"].median(),
        "p95_retrieval_s": df["retrieval_s"].quantile(0.95),
        "max_retrieval_s": df["retrieval_s"].max(),
    })

summary_df = pd.DataFrame(rows).sort_values("avg_retrieval_s")
summary_df.style.format({
    "avg_retrieval_s": "{:.3f}",
    "p50_retrieval_s": "{:.3f}",
    "p95_retrieval_s": "{:.3f}",
    "max_retrieval_s": "{:.3f}",
})